# Cookbook: applying the certificate to your own workflow

Each section below is a complete recipe for a specific situation. Find your scenario, copy the code, adapt the inputs to your own state / shadow data / catalog. All recipes run end-to-end without API keys; the adapter-dependent ones note which optional install they need.

**Recipe index**

| # | Situation | Use |
| --- | --- | --- |
| 1 | I have a converged PySCF / DFT mean-field state | `adapters.pyscf.from_mean_field` (zero-cost) |
| 2 | I have a post-HF state with 1-, 2-, 3-, 4-RDMs | `adapters.pyscf.from_rdms` (exact $\Delta$) |
| 3 | I have random-Pauli shadow data | `delta_ucb` (Hoeffding + Bonferroni UCB) |
| 4 | I have matchgate / fermionic-Gaussian shadow data | `delta_ucb_from_majorana_moments` (no JW penalty) |
| 5 | I want a custom catalog beyond `chemistry_r4` | `Catalog(words=..., r=...)` |
| 6 | I want to persist the certificate as a JSON artefact | `dataclasses.asdict` + `json.dump` |
| 7 | I want a go/no-go decision against a tolerance $\epsilon$ | compare bound to $\epsilon$ |
| 8 | Which `level` should I pick? | `block_refined` is the recommended default |
| 9 | I want to drop catalog observables into OpenFermion / Qiskit-Nature | `word_to_fermion_operator`, `word_to_fermionic_op` |

**The inequality at the core of every recipe** is the same:
$$
\bigl| \langle W \rangle_{\rho} - \langle W \rangle_{\mathrm{trunc}, \rho} \bigr|
\;\le\;
C_W \cdot \Delta_{r, U(1)}^{\mathrm{cat}}(\rho).
$$
All nine recipes differ only in how they produce $\Delta$ and which $C_W$ they plug in.

In [ ]:
%pip install -q "cumulant_residual_cert@git+https://github.com/kootru-repo/cumulant-residual-cert.git" numpy

## Recipe 1: I have a converged PySCF / DFT mean-field state

**Situation.** You have a converged Hartree-Fock or DFT calculation in PySCF (RHF, UHF, RKS, UKS). The state is a Slater determinant in the canonical molecular orbital basis. You want a certified bound on any cumulant-truncated chemistry-catalog observable derived from that state.

**Use.** `adapters.pyscf.from_mean_field`. The worked-example theorem proves $\Delta = 0$ exactly on this state class, so the certified bound is zero. No measurement needed.

**Optional install.** `pip install "cumulant_residual_cert[pyscf]@git+..."`


In [ ]:
# Skip this cell if you don't have PySCF installed; the recipe text above
# stands without running it.
try:
    from pyscf import gto, scf
    from cumulant_residual_cert.adapters.pyscf import from_mean_field
    from cumulant_residual_cert import Catalog

    mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g', verbose=0)
    mf = scf.RHF(mol).run(conv_tol=1e-10)

    catalog = Catalog.chemistry_r4()
    est = from_mean_field(mf, catalog, user_asserts_bernoulli_class=True)

    print(f'Delta              = {est.delta}')
    print(f'delta_is_exact     = {est.delta_is_exact}')
    print(f'delta_provenance   = {est.bound.delta_provenance}')
    print()
    print('Certified bias bars (all zero, exactly):')
    for word, bar in est.bound.bounds.items():
        print(f'  |bias({word:32s})| <= {bar}')
except ImportError:
    print('PySCF not installed; install with the [pyscf] extra to run this recipe.')

**Caveats.** The `user_asserts_bernoulli_class=True` argument is mandatory and means "yes, the dictionary basis I am certifying against is the canonical MO basis of `mf`, and the prepared state is the resulting Slater determinant." If your dictionary basis or prepared state is something else, do not use this helper; route through Recipe 2 (RDMs) instead.


## Recipe 2: I have a post-HF state with 1-, 2-, 3-, 4-RDMs

**Situation.** You ran CISD, CASCI, NEVPT2, FCI, or similar, and have the spin-orbital reduced density matrices in your hands as numpy arrays. You want $\Delta$ computed exactly from those RDMs.

**Use.** `adapters.pyscf.from_rdms`. The Mobius formula and the in-house normal-ordering routine evaluate every catalog-word connected cumulant directly from the RDMs.

**RDM convention.** $D^{(k)}[p_1,\ldots,p_k, q_1,\ldots,q_k] = \langle a^\dagger_{p_1} \cdots a^\dagger_{p_k} a_{q_k} \cdots a_{q_1} \rangle$. Indices are 0-based on the tensor.

In [ ]:
import numpy as np

from cumulant_residual_cert import Catalog
from cumulant_residual_cert.adapters.pyscf import from_rdms
from cumulant_residual_cert._fermion import letter_op
from itertools import product as iproduct

# Demo: build a small correlated state and its RDMs so the recipe is
# self-contained. In your own workflow you would replace this with
# rdms produced by PySCF's CISD / CASCI / FCI helpers.
n = 4
theta = np.pi / 5  # any non-trivial angle gives a correlated state
psi = np.zeros(2 ** n, dtype=complex)
psi[int('1100', 2)] = np.cos(theta)
psi[int('0011', 2)] = np.sin(theta)
rho = np.outer(psi, psi.conj())

def build_rdm(rho, k, n):
    shape = (n,) * (2 * k)
    rdm = np.zeros(shape, dtype=complex)
    for idx in iproduct(range(n), repeat=2 * k):
        p, q = idx[:k], idx[k:]
        op = letter_op('I', 1, n)
        for s0 in p:
            op = op @ letter_op('a_dag', s0 + 1, n)
        for s0 in reversed(q):
            op = op @ letter_op('a', s0 + 1, n)
        rdm[idx] = np.trace(rho @ op)
    return rdm

rdm1, rdm2, rdm3, rdm4 = (build_rdm(rho, k, n) for k in (1, 2, 3, 4))

catalog = Catalog.chemistry_r4()
sites_per_word = [
    (1, 2, 3), (1, 2, 3),
    (1, 2, 3, 4), (1, 2, 3, 4), (1, 2, 3, 4),
]
est = from_rdms(
    rdm1=rdm1, rdm2=rdm2, rdm3=rdm3, rdm4=rdm4,
    catalog=catalog, sites_per_word=sites_per_word,
)
print(f'Delta              = {est.delta:.6f}')
print(f'delta_provenance   = {est.bound.delta_provenance}    (expected from_rdms)')
print(f'library_version    = {est.bound.library_version}')
print()
print('Certified bounds at the block-refined level:')
for word, bar in est.bound.bounds.items():
    print(f'  |bias({word:32s})| <= {bar:.6f}')

**Caveats.** All RDM tensors must be in the same orbital basis, matching the sites you reference in `sites_per_word`. Length-3 catalog words need at least the 3-RDM; length-4 words need the 4-RDM. If you only have 1- and 2-RDM and the catalog has any length-$\ge 3$ word, the call raises `ValueError`. In that case route the missing length-$\ge 3$ subwords through Recipe 3 or 4 (shadow estimation).


## Recipe 3: I have random-Pauli shadow data

**Situation.** You have a list of `(basis, outcomes)` shadow shots from a random-Pauli measurement protocol. Each `basis` is a tuple of `'X'` / `'Y'` / `'Z'` on each of `n_qubits` sites; each `outcomes` is a tuple of $\pm 1$. You want a Bonferroni-corrected UCB on $\Delta$ at $1 - \alpha$ confidence.

**Use.** `delta_ucb`. It Bonferroni-corrects over every Pauli string in the catalog's subword expansions, then propagates Hoeffding radii through the Mobius transform.

**Caveat upfront.** The built-in expansion is dense in `n_qubits` and refuses to run past `n_qubits = 10`. For chemistry-scale registers, either bring your own per-Pauli estimates and call `delta_ucb_from_subword_moments` (Recipe 3b below), or use matchgate shadows (Recipe 4).

In [ ]:
from cumulant_residual_cert import Catalog, certify, delta_ucb
from cumulant_residual_cert.diagnostic import collect_shadows  # simulator helper

# Demo: simulate shadow data from the same correlated state as Recipe 2.
n = 4
shadows = collect_shadows(rho, n=n, M=1000, seed=42)

catalog = Catalog.chemistry_r4()
sites_per_word = [
    (1, 2, 3), (1, 2, 3),
    (1, 2, 3, 4), (1, 2, 3, 4), (1, 2, 3, 4),
]
ucb = delta_ucb(
    shadow_samples=shadows,
    catalog=catalog,
    sites_per_word=sites_per_word,
    n_qubits=n,
    confidence=0.95,
)
print(f'Delta UCB at 95% confidence = {ucb.delta_ucb:.4f}')
print(f'Paulis in Bonferroni union  = {ucb.n_paulis}')
print()
result = certify(catalog, delta=ucb.delta_ucb, delta_provenance='ucb_random_pauli')
print('Certified bounds:')
for word, bar in result.bounds.items():
    print(f'  |bias({word:32s})| <= {bar:.4f}')

**Recipe 3b: BYO Pauli estimates.** If you already have empirical $(\langle P \rangle, \mathrm{rad}_P)$ per Pauli string from your own protocol, build the per-subword moments yourself and route through `delta_ucb_from_subword_moments`. The benefit is that you bypass the dense `n_qubits \le 10` cap and can use any unbiased estimator + Hoeffding radius the protocol gives you.

## Recipe 4: I have matchgate / fermionic-Gaussian shadow data

**Situation.** You ran a matchgate-shadow measurement protocol (Wan-Hadfield-Cleve-Babbush 2022; Zhao-Rubin-Miyake-Babbush 2021). The output is an empirical mean $\hat{\langle \gamma_S \rangle}$ and a Hoeffding-style radius for every degree-$2k$ Majorana product $\gamma_S$ you measured. You want the matchgate range factor (which avoids the random-Pauli $3^{|P|}$ Jordan-Wigner penalty).

**Use.** `delta_ucb_from_majorana_moments` (or the OpenFermion adapter wrapper `delta_ucb_from_matchgate_shadows`). The library handles the dictionary-letter to Majorana decomposition and the Mobius assembly; the snapshot estimation is your responsibility.

**Status.** A built-in matchgate-snapshot estimator (Pfaffian + matchgate orthogonal-matrix algebra) is planned for a later release. Until then, you supply the Majorana moments yourself.

In [ ]:
from cumulant_residual_cert import Catalog, certify, delta_ucb_from_majorana_moments

# Demo: supply a few tight matchgate-style estimates. In your own
# workflow you would replace this dict with empirical (mean, radius)
# pairs produced by your matchgate-shadow code.
majorana_moments = {
    (): (1.0 + 0j, 0.0),         # identity always 1, exact
    (1, 2): (0.50 + 0j, 0.01),   # i/2 * gamma_1 gamma_2 = (n_1 - 1/2) etc.
    (3, 4): (0.25 + 0j, 0.01),
    (5, 6): (0.25 + 0j, 0.01),
    (7, 8): (0.50 + 0j, 0.01),
    # Higher-degree products. Library treats missing odd-degree entries
    # as zero (exact for U(1)-invariant states); supply explicit entries
    # for even-degree products you actually estimated.
}
catalog = Catalog.chemistry_r4()
sites_per_word = [
    (1, 2, 3), (1, 2, 3),
    (1, 2, 3, 4), (1, 2, 3, 4), (1, 2, 3, 4),
]
ucb = delta_ucb_from_majorana_moments(
    majorana_moments=majorana_moments,
    catalog=catalog,
    sites_per_word=sites_per_word,
    confidence=0.95,
    n_protocol_terms=len(majorana_moments),  # for the Bonferroni union
)
print(f'Delta UCB = {ucb.delta_ucb:.4f}')
result = certify(catalog, delta=ucb.delta_ucb, delta_provenance='ucb_matchgate_shadows')
print()
print('Certified bounds:')
for word, bar in result.bounds.items():
    print(f'  |bias({word:32s})| <= {bar:.4f}')

## Recipe 5: I want a custom catalog beyond `chemistry_r4`

**Situation.** Your application uses a different set of fermionic-word observables (a smaller chemistry subset, a non-chemistry list, a length-5 set for an extended catalog). You want the same `certify(...)` machinery on your own catalog.

**Use.** Build a `Catalog(words=..., r=...)` directly. The integer constants come from the partition lattice automatically.

In [ ]:
from cumulant_residual_cert import Catalog, FermionicWord, certify, constants

# Example: a smaller two-word catalog for a hopping + double-occupancy study.
my_catalog = Catalog(
    words=(
        FermionicWord(('a_dag', 'a', 'n'), name='hopping_with_density'),
        FermionicWord(('n', 'n', 'n'), name='triple_density'),
    ),
    r=4,           # max word length cap; you can certify words up to this length
    name='custom_hopping_density',
)
print(f'Custom catalog has {len(my_catalog)} words')
for w in my_catalog:
    Bu = constants.universal(my_catalog.r)
    Bc = constants.charge_filtered(my_catalog.r, w)
    Bh = constants.block_refined(my_catalog.r, w)
    print(f'  {w.name:24s}  universal={Bu:>3d}  charge={Bc:>3d}  block-refined={Bh:>3d}')
print()
result = certify(my_catalog, delta=0.02)
for word, bar in result.bounds.items():
    print(f'  |bias({word:24s})| <= {bar:.4f}')

**Caveats.** Words must be charge-neutral (`word.total_charge == 0`). Words longer than `r` are rejected. Catalog `r >= 3` only; the residual bound is undefined at $r < 3$.


## Recipe 6: Persist the certificate as a JSON artefact

**Situation.** You want to attach the certificate to your workflow output, your manuscript supplement, or your group's audit log. You need a serialisable, readable file that future auditors can interpret.

**Use.** `dataclasses.asdict` + `json.dump`. No library API is needed; the dataclass already carries the provenance fields a downstream reader requires (`library_version`, `delta_provenance`, `catalog_name`).

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from cumulant_residual_cert import Catalog, certify

catalog = Catalog.chemistry_r4()
result = certify(catalog, delta=0.012, delta_provenance='from_rdms')

artefact_path = Path('my_workflow_certificate.json')
artefact_path.write_text(json.dumps(asdict(result), indent=2))

print(f'Wrote {artefact_path} ({artefact_path.stat().st_size} bytes)')
print()
print('Round-trip read:')
loaded = json.loads(artefact_path.read_text())
for k in ('library_version', 'delta_provenance', 'catalog_name', 'level', 'delta'):
    print(f'  {k:18s} = {loaded[k]!r}')
print()
print(f'  bounds: {loaded["bounds"]}')

**Decoding the provenance label.** The `delta_provenance` field is the single most important field on the certificate, because a number is only as trustworthy as how it was obtained. Library-recognised values:

| label | meaning |
| --- | --- |
| `user_supplied` | Caller passed $\Delta$ directly; library did no further check. |
| `closed_form_bernoulli` | Adapter detected a Bernoulli-class state; $\Delta = 0$ exactly. |
| `from_rdms` | Mobius formula evaluated $\Delta$ from supplied RDMs exactly under U(1)-invariance. |
| `ucb_random_pauli` | Hoeffding + Bonferroni UCB from random-Pauli shadows. |
| `ucb_subword` | UCB from caller-supplied per-subword $(\hat\mu, \mathrm{rad})$. |
| `ucb_majorana` | UCB from caller-supplied per-Majorana-product $(\hat\mu, \mathrm{rad})$. |
| `ucb_matchgate_shadows` | UCB via OpenFermion matchgate-shadow wrapper. |

Custom labels are also accepted: the field type is free-form `str` so you can record "my-VQE-pipeline-shadow-protocol" or similar.

## Recipe 7: Go / no-go decision against a tolerance $\epsilon$

**Situation.** You have a target tolerance for the truncation bias: e.g., "the cumulant-truncated estimate must be within $\epsilon = 5 \times 10^{-3}$ of truth for downstream chemistry interpretation." You need a decision rule: is your truncated estimate trustworthy at that tolerance?

**Use.** Compare the certified bound to $\epsilon$ for the specific catalog word your workflow consumes.

In [ ]:
from cumulant_residual_cert import Catalog, certify

# Workflow inputs
epsilon = 5e-3                                # bias tolerance you decided on
Delta_estimate = 0.008                        # from Recipe 2, 3, or 4
target_word = 'n_i n_j n_k n_ell'             # the catalog word your workflow uses

catalog = Catalog.chemistry_r4()
result = certify(catalog, delta=Delta_estimate, level='block_refined')
bound = result.bounds[target_word]
print(f'Target word     : {target_word}')
print(f'Tolerance       : {epsilon:.2e}')
print(f'Certified bound : {bound:.2e}')
if bound <= epsilon:
    print(f'Decision: ACCEPT. Your truncated estimate is certified to within {epsilon:.2e}.')
else:
    print(f'Decision: REJECT. Bound exceeds tolerance by factor {bound/epsilon:.2f}.')
    print(f'  Options: (a) take more shadow data to tighten Delta;')
    print(f'           (b) supply higher-order RDMs to shrink Delta;')
    print(f'           (c) loosen epsilon if scientifically justified;')
    print(f'           (d) measure W directly instead of truncating.')

## Recipe 8: Which `level` should I pick?

**Default answer.** Use `level='block_refined'`. It is always the tightest of the three on any catalog word. There is no statistical or theoretical penalty for using it.

**When to use a looser level.**

- `charge_filtered` is useful if you are presenting a uniform bound across the catalog *and* the catalog has at least one word where the charge-filtered and block-refined constants coincide; the uniform bound is then easier to communicate.
- `universal` is useful as a single number $B_r$ that applies to **any** word of length $\le r$ on **any** $U(1)$-invariant state. Use it for sketching, scaling estimates, or worst-case argument-by-induction situations.

**Concrete comparison on the chemistry catalog at $r = 4$.**

In [ ]:
from cumulant_residual_cert import Catalog, constants

catalog = Catalog.chemistry_r4()
Bu = constants.universal(catalog.r)
print(f'{"word":32s} {"universal":>10s} {"charge":>10s} {"block-refined":>15s} {"refinement":>10s}')
print('-' * 80)
for w in catalog:
    Bc = constants.charge_filtered(catalog.r, w)
    Bh = constants.block_refined(catalog.r, w)
    print(f'{w.name:32s} {Bu:>10d} {Bc:>10d} {Bh:>15d} {Bu // Bh:>9d}x')

## Recipe 9: Use catalog observables with OpenFermion / Qiskit-Nature

**Situation.** You want to MEASURE one or more catalog observables on a quantum simulator or hardware, using an existing OpenFermion / Qiskit-Nature pipeline. You need the catalog words as `FermionOperator` / `FermionicOp` instances to plug into those frameworks' measurement / grouping primitives.

**Use.** `adapters.openfermion.word_to_fermion_operator` and `adapters.qiskit_nature.word_to_fermionic_op`. Both accept a `FermionicWord` plus a 1-based site assignment and return the framework-native operator.

**Optional installs.** `pip install "cumulant_residual_cert[openfermion]@git+..."` or `[qiskit-nature]@git+...`.

In [ ]:
from cumulant_residual_cert import Catalog

catalog = Catalog.chemistry_r4()
hopping_word = next(w for w in catalog if w.name == 'a_dag_i a_j n_k')
sites = (1, 2, 3)

try:
    from cumulant_residual_cert.adapters.openfermion import word_to_fermion_operator
    fop = word_to_fermion_operator(hopping_word, sites)
    print('OpenFermion FermionOperator (drop into openfermion.measurements):')
    print(fop)
except ImportError:
    print('OpenFermion not installed; install with the [openfermion] extra.')

print()

try:
    from cumulant_residual_cert.adapters.qiskit_nature import word_to_fermionic_op
    qop = word_to_fermionic_op(hopping_word, sites)
    print('Qiskit-Nature FermionicOp (drop into qiskit_nature estimators):')
    print(qop)
except ImportError:
    print('qiskit-nature not installed; install with the [qiskit-nature] extra.')

## Mixing recipes

The recipes compose. A realistic chemistry workflow often uses several at once:

- **HF + post-HF in the same pipeline.** Use Recipe 1 to get the zero-cost HF certificate, then route post-HF excitations through Recipe 2 (RDMs) or Recipes 3 / 4 (shadows). Concatenate the certificates.
- **Adaptive shadow protocol.** Use Recipe 3 as the initial measurement; if Recipe 7's go / no-go fails, escalate to Recipe 4 (matchgate shadows) to tighten Delta at the same shot count.
- **Custom catalog + custom estimator.** Combine Recipe 5 with `delta_ucb_from_subword_moments` (BYO per-subword estimates) for measurement protocols the library does not have a built-in path for.

In every case the persisted certificate from Recipe 6 carries the `delta_provenance` label so a downstream auditor can tell which path produced the bound.